# Machine Sound Fault Detection (MIMII dataset)

This notebook finds **faulty machines by their sound**.

**How it works**
1. We train a model on **healthy** sound only.
2. The model learns to copy healthy sound.
3. A faulty sound is copied badly. A **big copy error = possible fault**.

**What you need to do**
1. Add the MIMII dataset to this notebook (**Add Input**).
2. Turn on the GPU: **Settings > Accelerator > GPU**.
3. Click **Run All**.

**Default run:** 4 machines (fan, pump, slider, valve) x up to 7 IDs x 6 dB noise. It uses the **dense model** only.
Time: about 2 hours on a GPU (my guess from tests on a home GPU; Kaggle can be different).

## Words used in this notebook

| Word | Meaning |
|---|---|
| **clip** | One 10-second sound file |
| **window** | A 2-second piece of a clip |
| **healthy / faulty** | Machine works well / machine has a problem |
| **score** | The copy error of a clip. Higher = more likely faulty |
| **AUC** | How well the score separates healthy from faulty. **1.0 = perfect, 0.5 = random guess** |
| **pAUC** | Same as AUC, but only when false alarms are very low (under 10%). 0.5 = random guess |
| **F1** | One number for how good the alarm is at one chosen threshold. 1.0 = perfect |
| **threshold** | The score above which we say "faulty" |

## Step 1 - Import tools

In [ ]:
import glob, gc, re, sys, time, subprocess, importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from joblib import Parallel, delayed
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from IPython.display import display

# librosa and soundfile are usually on Kaggle already. Install them only if missing.
for name in ("librosa", "soundfile"):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
import librosa
import soundfile as sf

## Step 2 - Settings

Change the settings here. The defaults are fine.

In [ ]:
# ---- Where things are ----
DATA_ROOT = "/kaggle/input"      # the audio files are somewhere inside this folder
WORK_DIR  = "/kaggle/working"    # results are saved here

# ---- What to run ----
MACHINES  = ["fan", "pump", "slider", "valve"]
IDS       = ["id_00", "id_01", "id_02", "id_03", "id_04", "id_05", "id_06"]
SNR       = 6                    # noise level: 6, 0 or -6
MAX_HOURS = 10                   # stop starting new runs after this many hours

# ---- Sound settings ----
SAMPLE_RATE = 16000
N_FFT, HOP, N_MELS = 1024, 512, 128
MIC     = 0                      # the dataset has 8 microphones. We use the first one.
WINDOW  = 32768                  # samples per window (about 2 seconds)
STEP    = 16384                  # the next window starts 1 second later
FRAMES  = 64                     # columns kept in each picture

# ---- Model settings ----
STACK     = 5                    # how many neighbour columns the model sees together
LATENT    = 16                   # size of the small middle layer
LAYERS    = [1024, 512, 256, 128]
BATCH     = 512
TRAIN_STEP = 2                   # use every 2nd column to save memory
LEARNING_RATE = 1e-3
EPOCHS    = 100
THRESHOLD_PERCENT = 95           # threshold = 95th percentile of healthy validation scores
SEED      = 42

## Step 3 - Check the GPU

You should see one GPU. If the list is empty, turn on the GPU in **Settings > Accelerator**.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("TensorFlow", tf.__version__)
print("GPUs found:", [g.name for g in gpus])

## Step 4 - Find the audio files

We look for every `.wav` file. The folder names tell us the **machine**, **ID**, **healthy/faulty** and **noise level**.

In [ ]:
def read_path(path):
    parts = path.split("/")[::-1]          # start from the file name
    label   = next((p for p in parts if p in ("normal", "abnormal")), None)
    machine = next((p for p in parts if p in ("fan", "pump", "slider", "valve")), None)
    mid     = next((p for p in parts if re.fullmatch(r"id_\d+", p)), None)
    snr = None
    for p in parts:
        m = re.match(r"(-?\d+)\s*_?dB", p)
        if m:
            snr = int(m.group(1))
            break
    return label, machine, mid, snr

rows = []
for path in glob.glob(DATA_ROOT + "/**/*.wav", recursive=True):
    label, machine, mid, snr = read_path(path)
    if label and machine and mid:
        rows.append([path, machine, mid, label, snr])

df = pd.DataFrame(rows, columns=["path", "machine", "id", "label", "snr"])
assert len(df) > 0, "No audio found. Did you add the dataset to this notebook (Add Input)?"
print(len(df), "clips found")

Number of clips for each machine, ID and noise level:

In [ ]:
counts = df.groupby(["machine", "id", "snr", "label"], dropna=False).size().unstack("label", fill_value=0)
display(counts)

## Step 5 - Choose the runs

One **run** = one machine + one ID. We skip a run if it has too few clips.

In [ ]:
USE_SNR = df["snr"].notna().any()     # some datasets have no noise level in the folder name

def get_files(machine, mid):
    rows = df[(df["machine"] == machine) & (df["id"] == mid)]
    if USE_SNR:
        rows = rows[rows["snr"] == SNR]
    healthy = sorted(rows[rows["label"] == "normal"]["path"])
    faulty  = sorted(rows[rows["label"] == "abnormal"]["path"])
    return healthy, faulty

plan = []
for machine in MACHINES:
    for mid in IDS:
        healthy, faulty = get_files(machine, mid)
        if len(healthy) >= 30 and len(faulty) >= 5:
            plan.append((machine, mid))
        else:
            print("skip", machine, mid, "- not enough clips")

print(len(plan), "runs planned:")
print(plan)

## Step 6 - Turn sound into a picture

A model cannot read a sound file easily. So we make a picture of the sound.
This picture is called a **log-mel spectrogram**: time goes left to right, pitch goes bottom to top, and brightness shows loudness.

In [ ]:
def load_audio(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)   # shape: (samples, microphones)
    audio = audio[:, min(MIC, audio.shape[1] - 1)]                 # keep one microphone
    if rate != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=rate, target_sr=SAMPLE_RATE)
    return audio

def cut_windows(audio):
    if len(audio) < WINDOW:
        audio = np.pad(audio, (0, WINDOW - len(audio)))
    starts = range(0, len(audio) - WINDOW + 1, STEP)
    return [audio[s:s + WINDOW] for s in starts]

def to_logmel(window):
    mel = librosa.feature.melspectrogram(y=window, sr=SAMPLE_RATE, n_fft=N_FFT,
                                         hop_length=HOP, n_mels=N_MELS)
    db = librosa.power_to_db(mel, ref=1.0)
    return db[:, :FRAMES]                                          # shape: (128, 64)

Look at one healthy clip and one faulty clip:

In [ ]:
machine, mid = plan[0]
healthy, faulty = get_files(machine, mid)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, path, name in zip(axs, [healthy[0], faulty[0]], ["healthy", "faulty"]):
    ax.imshow(to_logmel(load_audio(path)[:WINDOW]), origin="lower", aspect="auto", cmap="magma")
    ax.set(title=f"{machine} {mid} - {name}", xlabel="time", ylabel="mel band")
plt.show()

## Step 7 - Load the clips

For each clip we make many windows. Then we make one picture for each window.
This step uses all CPU cores to go faster.

In [ ]:
def clip_to_pictures(path):
    windows = cut_windows(load_audio(path))
    return np.array([to_logmel(w) for w in windows], dtype=np.float32)

def load_clips(files):
    pieces = Parallel(n_jobs=-1)(delayed(clip_to_pictures)(f) for f in files)
    pictures = np.concatenate(pieces)                                          # all windows together
    clip_id = np.concatenate([np.full(len(p), i) for i, p in enumerate(pieces)])  # which clip each window came from
    return pictures, clip_id

## Step 8 - Split the data

- **Train**: 70% of healthy clips. The model learns from these.
- **Validation**: 10% of healthy clips. We use them to stop training and to set the threshold.
- **Test**: the other 20% of healthy clips + **all** faulty clips. We only use them to measure the result.

We split by **clip** (not by window). So windows from one clip never go to two groups. This avoids cheating.

In [ ]:
def split_healthy(healthy_files):
    order = np.random.default_rng(SEED).permutation(len(healthy_files))
    n_train = int(0.7 * len(order))
    n_val   = int(0.1 * len(order))
    train = [healthy_files[i] for i in order[:n_train]]
    val   = [healthy_files[i] for i in order[n_train:n_train + n_val]]
    test  = [healthy_files[i] for i in order[n_train + n_val:]]
    return train, val, test

## Step 9 - Make the model input

The model does not look at one column of the picture. It looks at **5 neighbour columns together** (5 x 128 = 640 numbers).

We also scale each pitch row: subtract its average and divide by its spread.
The average and spread come from the **train** set only.

In [ ]:
def to_vectors(pictures, mean, std, step=1):
    scaled = (pictures - mean) / std                                          # (windows, 128, 64)
    stacked = np.lib.stride_tricks.sliding_window_view(scaled, STACK, axis=2)  # (windows, 128, 60, 5)
    stacked = stacked[:, :, ::step].transpose(0, 2, 3, 1)                     # (windows, 60, 5, 128)
    return stacked.reshape(len(pictures), -1, STACK * N_MELS).astype(np.float32)

## Step 10 - The model (dense autoencoder)

The model has two parts:
- **Encoder**: squeezes the 640 numbers down to only 16 numbers.
- **Decoder**: tries to rebuild the 640 numbers from the 16.

Only 16 numbers are not enough to copy every detail. So the model learns only the **normal** pattern.

In [ ]:
def build_model():
    inputs = tf.keras.Input(shape=(STACK * N_MELS,))
    x = inputs
    for size in LAYERS:                                   # encoder
        x = tf.keras.layers.Dense(size)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Dense(LATENT, name="middle")(x)   # the small middle layer
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    for size in reversed(LAYERS):                         # decoder
        x = tf.keras.layers.Dense(size)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
    outputs = tf.keras.layers.Dense(STACK * N_MELS)(x)
    return tf.keras.Model(inputs, outputs)

build_model().summary()

## Step 11 - Train the model

Training stops by itself when the validation loss stops getting better (**early stopping**).
The learning rate also gets smaller when training is stuck.

In [ ]:
def make_dataset(vectors, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(vectors)
    if shuffle:
        ds = ds.shuffle(50000, seed=SEED)
    ds = ds.map(lambda x: (x, x))              # the target is the input itself
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

def train_model(train_vectors, val_vectors):
    model = build_model()
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="mse")
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    ]
    history = model.fit(make_dataset(train_vectors, shuffle=True),
                        validation_data=make_dataset(val_vectors),
                        epochs=EPOCHS, callbacks=callbacks, verbose=0)
    return model, history.history

## Step 12 - Score the clips

**Score = copy error.** We rebuild each clip with the model and measure the mistake (MSE).
Each window gets a score. A clip score is the **average** of its window scores.

In [ ]:
def clip_scores(model, pictures, clip_id, mean, std, chunk=64):
    window_scores = []
    for i in range(0, len(pictures), chunk):
        vectors = to_vectors(pictures[i:i + chunk], mean, std)     # (windows, 60, 640)
        flat = vectors.reshape(-1, STACK * N_MELS)
        rebuilt = model.predict(flat, batch_size=4096, verbose=0)
        error = np.mean((flat - rebuilt) ** 2, axis=1)             # error of each column group
        window_scores.append(error.reshape(len(vectors), -1).mean(axis=1))
    window_scores = np.concatenate(window_scores)
    return np.array([window_scores[clip_id == c].mean() for c in np.unique(clip_id)])

## Step 13 - Measure the result

- **Threshold** = 95th percentile of healthy **validation** scores. So about 5% of healthy clips raise a false alarm.
- We never use faulty clips to set the threshold.

In [ ]:
def get_metrics(val_scores, test_scores, y_test):
    threshold = np.percentile(val_scores, THRESHOLD_PERCENT)
    predicted = (test_scores > threshold).astype(int)
    return {
        "auc":  roc_auc_score(y_test, test_scores),
        "pauc": roc_auc_score(y_test, test_scores, max_fpr=0.1),
        "f1":   f1_score(y_test, predicted),
        "threshold": threshold,
    }

## Step 14 - Save files for later

After each run we save **everything** into `/kaggle/working/outputs/<machine>_<id>/`:

| File | What it holds |
|---|---|
| `model.keras` | the trained model (weights + layout) |
| `norm.npz` | `mean` and `std` used for scaling (needed to reuse the model) |
| `clip_scores.csv` | one row per clip: file name, true label, score, alarm yes/no |
| `history.csv` | loss per epoch |
| `metrics.json` | AUC, pAUC, F1, threshold |

Use these to analyse the results later, or to score new audio **without training again**.

In [ ]:
import json, os

OUT_DIR = WORK_DIR + "/outputs"

def save_run(machine, mid, model, mean, std, val_files, test_files, saved):
    folder = f"{OUT_DIR}/{machine}_{mid}"
    os.makedirs(folder, exist_ok=True)

    model.save(folder + "/model.keras")
    np.savez(folder + "/norm.npz", mean=mean, std=std)

    threshold = saved["metrics"]["threshold"]
    test_table = pd.DataFrame({
        "machine": machine, "id": mid, "snr": SNR, "split": "test",
        "file": [str(f) for f in test_files],
        "label": saved["y_test"],                              # 1 = faulty
        "score": saved["test_scores"],
        "alarm": (saved["test_scores"] > threshold).astype(int),
    })
    val_table = pd.DataFrame({
        "machine": machine, "id": mid, "snr": SNR, "split": "val",
        "file": [str(f) for f in val_files],
        "label": 0,
        "score": saved["val_scores"],
        "alarm": (saved["val_scores"] > threshold).astype(int),
    })
    pd.concat([val_table, test_table]).to_csv(folder + "/clip_scores.csv", index=False)

    pd.DataFrame(saved["history"]).to_csv(folder + "/history.csv", index_label="epoch")
    with open(folder + "/metrics.json", "w") as f:
        json.dump({k: float(v) for k, v in saved["metrics"].items()}, f, indent=2)

## Step 15 - One full run

This function does everything for one machine + one ID:
load, split, train, score, measure.

It also saves the scores and the loss curve, so we can draw charts later **without training again**.

In [ ]:
# Published AUC from the MIMII paper (Table 3), for comparison.
# Order inside each tuple: 6 dB, 0 dB, -6 dB
PUBLISHED = {
 "fan":    {"id_00": (.75,.63,.57), "id_01": (.97,.90,.70), "id_02": (.99,.83,.68), "id_03": (1.0,.89,.70),
            "id_04": (.92,.75,.57), "id_05": (.95,.90,.83), "id_06": (.99,.97,.83)},
 "pump":   {"id_00": (.84,.65,.58), "id_01": (.98,.90,.73), "id_02": (.45,.46,.52), "id_03": (.79,.81,.75),
            "id_04": (.99,.95,.93), "id_05": (.66,.66,.64), "id_06": (.94,.76,.61)},
 "slider": {"id_00": (.99,.99,.93), "id_01": (.94,.90,.83), "id_02": (.93,.79,.74), "id_03": (.99,.85,.71),
            "id_04": (.88,.78,.61), "id_05": (.84,.70,.60), "id_06": (.71,.56,.52)},
 "valve":  {"id_00": (.68,.55,.62), "id_01": (.77,.71,.61), "id_02": (.66,.59,.57), "id_03": (.70,.65,.44),
            "id_04": (.64,.65,.50), "id_05": (.52,.48,.44), "id_06": (.70,.66,.53)},
}
SNR_COLUMN = {6: 0, 0: 1, -6: 2}

def run_one(machine, mid):
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)

    # 1. files and split
    healthy, faulty = get_files(machine, mid)
    train_f, val_f, test_healthy_f = split_healthy(healthy)
    test_f = test_healthy_f + faulty
    y_test = np.array([0] * len(test_healthy_f) + [1] * len(faulty))     # 1 = faulty
    print(f"  clips: train {len(train_f)}, val {len(val_f)}, test {len(test_healthy_f)} healthy + {len(faulty)} faulty")

    # 2. sound -> pictures
    X_train, _          = load_clips(train_f)
    X_val,  val_clip_id = load_clips(val_f)
    X_test, test_clip_id = load_clips(test_f)

    # 3. scaling numbers come from the train set only
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std  = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

    # 4. train
    train_vectors = to_vectors(X_train, mean, std, step=TRAIN_STEP).reshape(-1, STACK * N_MELS)
    val_vectors   = to_vectors(X_val,   mean, std, step=TRAIN_STEP).reshape(-1, STACK * N_MELS)
    model, history = train_model(train_vectors, val_vectors)

    # 5. score and measure
    val_scores  = clip_scores(model, X_val,  val_clip_id,  mean, std)
    test_scores = clip_scores(model, X_test, test_clip_id, mean, std)
    metrics = get_metrics(val_scores, test_scores, y_test)

    published = PUBLISHED[machine][mid][SNR_COLUMN[SNR]] if SNR in SNR_COLUMN else np.nan
    row = {"machine": machine, "id": mid, "snr": SNR, "auc": metrics["auc"], "pauc": metrics["pauc"],
           "f1": metrics["f1"], "published_auc": published, "epochs": len(history["loss"])}
    saved = {"history": history, "val_scores": val_scores, "test_scores": test_scores,
             "y_test": y_test, "metrics": metrics}
    save_run(machine, mid, model, mean, std, val_f, test_f, saved)
    return row, saved

## Step 16 - Run everything

This is the long step. Results are saved to `results.csv` after **every** run.
If Kaggle stops, you do not lose the finished runs.

In [ ]:
results = []        # one row per run
SAVED = {}          # scores and loss curves for the charts
start = time.time()

for number, (machine, mid) in enumerate(plan, 1):
    if (time.time() - start) / 3600 > MAX_HOURS:
        print("Time limit reached. Stopping.")
        break
    print(f"[{number}/{len(plan)}] {machine} {mid}")
    row, saved = run_one(machine, mid)
    results.append(row)
    SAVED[(machine, mid)] = saved
    pd.DataFrame(results).to_csv(WORK_DIR + "/results.csv", index=False)
    print(f"  AUC {row['auc']:.3f} | pAUC {row['pauc']:.3f} | F1 {row['f1']:.3f} | published AUC {row['published_auc']} | {row['epochs']} epochs")

print(f"Done in {(time.time() - start) / 60:.1f} minutes")

## Step 17 - Collect all outputs

- `all_clip_scores.csv` = every clip of every run in one table.
- `outputs.zip` = all folders zipped, easy to download from the Output tab.

In [ ]:
import glob, shutil

parts = [pd.read_csv(f) for f in sorted(glob.glob(OUT_DIR + "/*/clip_scores.csv"))]
all_scores = pd.concat(parts, ignore_index=True)
all_scores.to_csv(WORK_DIR + "/all_clip_scores.csv", index=False)
print("all_clip_scores.csv:", all_scores.shape)

shutil.make_archive(WORK_DIR + "/outputs", "zip", OUT_DIR)
print("outputs.zip written")
all_scores.head()

## Results

### Table of all runs

In [ ]:
res = pd.DataFrame(results)
display(res.round(3))

### Average per machine

Compare **auc** with **published_auc**. If auc is higher, we beat the paper.

In [ ]:
display(res.groupby("machine")[["auc", "pauc", "f1", "published_auc"]].mean().round(3))

## Charts for all runs

### Colour maps

Green = good. Red = bad. Rows are machines, columns are IDs.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, (column, title, low) in zip(axs, [("auc", "AUC", 0.5), ("pauc", "pAUC", 0.5), ("f1", "F1", 0.0)]):
    table = res.pivot_table(index="machine", columns="id", values=column)
    picture = ax.imshow(table.values, vmin=low, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(table.shape[1])); ax.set_xticklabels(table.columns)
    ax.set_yticks(range(table.shape[0])); ax.set_yticklabels(table.index)
    ax.grid(False)
    for (i, j), value in np.ndenumerate(table.values):
        if not np.isnan(value):
            ax.text(j, i, f"{value:.2f}", ha="center", va="center")
    ax.set_title(title)
    fig.colorbar(picture, ax=ax)
plt.tight_layout()
plt.show()

### Our AUC vs the published AUC

In [ ]:
x = np.arange(len(res))
plt.figure(figsize=(max(9, 0.6 * len(res)), 4.5))
plt.bar(x - 0.2, res["auc"], width=0.4, label="ours")
plt.bar(x + 0.2, res["published_auc"], width=0.4, label="published (MIMII paper)")
plt.axhline(0.5, color="gray", linestyle="--")
plt.xticks(x, res["machine"] + "\n" + res["id"], fontsize=8)
plt.ylabel("AUC"); plt.ylim(0, 1.05); plt.legend(loc="lower right")
plt.show()

## Charts for one run

Pick one run. The next cells draw charts for it.

Good runs to look at: the best and the worst, for example `("valve", "id_00")` and `("pump", "id_02")`.

In [ ]:
KEY = ("valve", "id_00")          # <-- change this
if KEY not in SAVED:
    KEY = list(SAVED)[0]
    print("That run was not found. Using", KEY)

run = SAVED[KEY]
history, metrics = run["history"], run["metrics"]
val_scores, test_scores, y_test = run["val_scores"], run["test_scores"], run["y_test"]
threshold = metrics["threshold"]
print("Run:", KEY, "| AUC", round(metrics["auc"], 3), "| pAUC", round(metrics["pauc"], 3), "| F1", round(metrics["f1"], 3))

### Loss curve

The loss is the copy error during training. Both lines should go down. If **validation** goes up while **train** goes down, the model is memorizing.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["loss"], label="train")
plt.plot(history["val_loss"], label="validation")
plt.yscale("log")
plt.xlabel("epoch"); plt.ylabel("loss (MSE)")
plt.title(f"Loss - {KEY[0]} {KEY[1]}")
plt.legend()
plt.show()

### ROC curve

The curve shows: for each false alarm rate, how many faulty clips we catch.
The **orange area** is where false alarms are under 10%. A curve far above the dashed line is good.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_scores)
plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, label=f"AUC = {metrics['auc']:.3f}   pAUC = {metrics['pauc']:.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.axvspan(0, 0.1, color="orange", alpha=0.2, label="pAUC area")
plt.xlabel("false alarm rate"); plt.ylabel("faulty clips caught")
plt.title("ROC curve")
plt.legend(loc="lower right")
plt.show()

### Score distribution

Healthy clips should have **low** scores. Faulty clips should have **high** scores.
The less the two shapes overlap, the better the model.
The dashed line is the threshold.

In [ ]:
all_scores = np.concatenate([val_scores, test_scores])
bins = np.linspace(all_scores.min(), all_scores.max(), 40)

plt.figure(figsize=(8, 4))
plt.hist(val_scores, bins, alpha=0.4, density=True, color="gray", label="healthy (validation)")
plt.hist(test_scores[y_test == 0], bins, alpha=0.5, density=True, color="tab:green", label="healthy (test)")
plt.hist(test_scores[y_test == 1], bins, alpha=0.5, density=True, color="tab:red", label="faulty (test)")
plt.axvline(threshold, color="black", linestyle="--", label="threshold")
plt.xlabel("clip score (copy error)"); plt.ylabel("density")
plt.legend()
plt.show()

### Confusion matrix

Each box counts clips.
- Top-left: healthy, and we said healthy (correct).
- Bottom-right: faulty, and we said faulty (correct).
- Top-right: healthy, but we said faulty (**false alarm**).
- Bottom-left: faulty, but we said healthy (**missed fault**).

In [ ]:
predicted = (test_scores > threshold).astype(int)
cm = confusion_matrix(y_test, predicted)
ConfusionMatrixDisplay(cm, display_labels=["healthy", "faulty"]).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion matrix at the {THRESHOLD_PERCENT}th percentile threshold")
plt.grid(False)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Correct: {tn + tp} | Wrong: {fp + fn}  (false alarms {fp}, missed faults {fn})")

### Accuracy for every threshold

An autoencoder has no "accuracy" while training. It only has accuracy **after** we choose a threshold.
So here we try many thresholds. The dashed line is the one we use.

**Balanced accuracy** is fairer when there are more faulty clips than healthy clips.

In [ ]:
thresholds = np.quantile(all_scores, np.linspace(0, 1, 200))
is_faulty = y_test == 1
accuracy, balanced, recall = [], [], []

for t in thresholds:
    said_faulty = test_scores > t
    tp = np.sum(said_faulty & is_faulty)
    tn = np.sum(~said_faulty & ~is_faulty)
    accuracy.append((tp + tn) / len(y_test))
    recall.append(tp / is_faulty.sum())                       # faulty clips caught
    balanced.append((tp / is_faulty.sum() + tn / (~is_faulty).sum()) / 2)

plt.figure(figsize=(8, 4))
plt.plot(thresholds, accuracy, label="accuracy")
plt.plot(thresholds, balanced, label="balanced accuracy")
plt.plot(thresholds, recall, ":", label="faulty clips caught")
plt.axvline(threshold, color="black", linestyle="--", label="our threshold")
plt.xlabel("threshold"); plt.ylabel("score"); plt.ylim(0, 1.02)
plt.legend()
plt.show()

### Scores by group

In [ ]:
groups = [val_scores, test_scores[y_test == 0], test_scores[y_test == 1]]
plt.figure(figsize=(6, 4))
plt.boxplot(groups, showfliers=False)
plt.axhline(threshold, color="black", linestyle="--")
plt.xticks([1, 2, 3], ["healthy\n(validation)", "healthy\n(test)", "faulty\n(test)"])
plt.yscale("log"); plt.ylabel("clip score")
plt.show()

## How many mistakes at each threshold?

A lower percentile = lower threshold = **more false alarms** but **fewer missed faults**.

Do not choose the threshold by looking at the test result. That is cheating.
Choose it by how many false alarms you can accept.

In [ ]:
rows = []
for percent in (99, 95, 90, 80, 70):
    t = np.percentile(val_scores, percent)
    said_faulty = test_scores > t
    false_alarms = int(np.sum(said_faulty & ~is_faulty))
    missed = int(np.sum(~said_faulty & is_faulty))
    rows.append([f"P{percent}", false_alarms, missed, false_alarms + missed])

display(pd.DataFrame(rows, columns=["threshold", "false alarms", "missed faults", "total wrong"]))

## Notes

- This notebook uses only the **dense model**. The convolutional model is in the local notebook.
- Our numbers are compared with the MIMII paper. The test split is not exactly the same, so treat the comparison as approximate.
- One run uses one random seed. A change of about 0.02 in AUC can be just luck.
- If faulty clips were recorded at a different time than healthy clips, the model may learn the **recording difference** and not the fault. We did not test this.